# Exercice 2 — Chunking et évaluation de la récupération

Ce notebook compare trois stratégies de découpage sur un texte français, puis évalue la recherche BM25 avec huit questions annotées manuellement.

Le texte d’exemple du dépôt (`data/text_samples/texte_exemple_rag.txt`) fait plus de 1 500 mots. Le notebook en prend les paragraphes jusqu’à dépasser légèrement 1 500 mots. Aucune clé API ni embedding n’est utilisé.

> Dépendances utilisées : `rank_bm25` est dans `requirements.txt`; `pandas` y est déjà présent.


## 1. Charger le texte et les questions annotées

Pour chaque question, `passage_reference` contient un extrait exact du texte qui porte la réponse. Un chunk est considéré pertinent s’il contient cet extrait. C’est une vérité terrain simple et vérifiable à la main.


In [ ]:
from pathlib import Path
import re
import pandas as pd
from rank_bm25 import BM25Okapi

# Le chemin fonctionne si le notebook est lancé depuis la racine du projet
# ou depuis le dossier notebooks.
candidats = [
    Path("data/text_samples/texte_exemple_rag.txt"),
    Path("../data/text_samples/texte_exemple_rag.txt"),
]
chemin_texte = next((p for p in candidats if p.exists()), None)
if chemin_texte is None:
    raise FileNotFoundError("Impossible de trouver data/text_samples/texte_exemple_rag.txt")

texte_complet = chemin_texte.read_text(encoding="utf-8")
paragraphes = [p.strip() for p in texte_complet.split("\n\n") if p.strip()]
paragraphes_selectionnes = []
nombre_mots = 0
for paragraphe in paragraphes:
    paragraphes_selectionnes.append(paragraphe)
    nombre_mots += len(paragraphe.split())
    if nombre_mots >= 1500:
        break

texte = "\n\n".join(paragraphes_selectionnes)
print(f"Fichier : {chemin_texte}")
print(f"Taille de l’extrait : {nombre_mots} mots")
print(f"Taille de l’extrait : {len(texte)} caractères")


In [ ]:
jeu_test = [
    {
        "question": "D’où peuvent provenir les données ingérées dans un pipeline RAG ?",
        "passage_reference": "récupérer les données depuis plusieurs sources",
    },
    {
        "question": "Donnez un exemple de problème que le nettoyage du texte doit corriger.",
        "passage_reference": "des espaces multiples",
    },
    {
        "question": "Quelles métadonnées peuvent accompagner un document ?",
        "passage_reference": "source, page, auteur, date, type",
    },
    {
        "question": "Quelle taille de chunk est utilisée dans l’exemple de code ?",
        "passage_reference": "chunk_size=1000",
    },
    {
        "question": "Quel modèle d’embedding est cité dans le mini-pipeline ?",
        "passage_reference": "sentence-transformers/all-MiniLM-L6-v2",
    },
    {
        "question": "Quel avantage de coût la RAG apporte-t-elle par rapport à l’ajustement ?",
        "passage_reference": "une approche plus rentable pour personnaliser les LLM",
    },
    {
        "question": "Que permettent les moteurs d’IA intégrés des processeurs Xeon ?",
        "passage_reference": "moteurs d'IA intégrés pour accélérer les opérations clés du pipeline",
    },
    {
        "question": "Quelles technologies Intel sont citées pour le traitement sécurisé ?",
        "passage_reference": "Intel® Software Guard Extensions (Intel® SGX) et Intel® Trust Domain Extensions (Intel® TDX)",
    },
]

# Vérification des annotations : chaque référence doit être présente dans le texte choisi.
for exemple in jeu_test:
    assert exemple["passage_reference"] in texte, exemple["passage_reference"]

print(f"Questions annotées : {len(jeu_test)}")


## 2. Implémenter les stratégies de découpage

- **A** : blocs de 200 caractères, sans chevauchement.
- **B** : blocs de 500 caractères, avec 100 caractères de chevauchement.
- **C** : regrouper phrases et paragraphes jusqu’à environ 400 caractères. Une phrase plus longue est coupée en segments de 400 caractères.


In [ ]:
def decouper_taille_fixe(texte, taille, chevauchement=0):
    pas = taille - chevauchement
    if pas <= 0:
        raise ValueError("Le chevauchement doit être inférieur à la taille du chunk.")
    return [
        texte[debut:debut + taille].strip()
        for debut in range(0, len(texte), pas)
        if texte[debut:debut + taille].strip()
    ]

def decouper_paragraphes_phrases(texte, taille_max=400):
    paragraphes = [p.strip() for p in texte.split("\n\n") if p.strip()]
    morceaux = []
    courant = ""

    def ajouter_phrase(phrase):
        nonlocal courant
        phrase = phrase.strip()
        if not phrase:
            return
        if len(phrase) > taille_max:
            if courant:
                morceaux.append(courant)
                courant = ""
            morceaux.extend(
                phrase[i:i + taille_max].strip()
                for i in range(0, len(phrase), taille_max)
                if phrase[i:i + taille_max].strip()
            )
        elif not courant:
            courant = phrase
        elif len(courant) + 1 + len(phrase) <= taille_max:
            courant += " " + phrase
        else:
            morceaux.append(courant)
            courant = phrase

    for paragraphe in paragraphes:
        phrases = re.split(r"(?<=[.!?])\s+|\n+", paragraphe)
        for phrase in phrases:
            ajouter_phrase(phrase)
        # Garde une frontière entre paragraphes sans forcer un chunk vide.
        if courant:
            morceaux.append(courant)
            courant = ""

    if courant:
        morceaux.append(courant)
    return morceaux

strategies = {
    "A — 200 caractères": decouper_taille_fixe(texte, taille=200),
    "B — 500 caractères, overlap 100": decouper_taille_fixe(
        texte, taille=500, chevauchement=100
    ),
    "C — phrases/paragraphes, ~400 caractères": decouper_paragraphes_phrases(
        texte, taille_max=400
    ),
}

for nom, chunks in strategies.items():
    print(f"{nom} : {len(chunks)} chunks")


## 3. Indexer chaque stratégie et mesurer Recall@3 et MRR

La recherche est faite avec BM25. Un chunk est pertinent s’il contient le passage de référence de la question. Pour chaque question, on prend le rang du premier chunk pertinent : Recall@3 vaut 1 si ce rang est au plus 3, et le reciprocal rank vaut `1 / rang` (ou 0 si aucun chunk pertinent n’est retrouvé).


In [ ]:
def tokenizer(texte):
    return re.findall(r"\w+", texte.casefold(), flags=re.UNICODE)

def evaluer(chunks, questions=jeu_test, reranking=False):
    corpus_tokenise = [tokenizer(chunk) for chunk in chunks]
    index = BM25Okapi(corpus_tokenise)
    recall_hits = []
    reciprocal_ranks = []

    for exemple in questions:
        tokens_question = tokenizer(exemple["question"])
        scores = index.get_scores(tokens_question)
        if reranking:
            # Bonus simple : +1 si tous les mots de la question apparaissent dans le chunk.
            for i, chunk in enumerate(chunks):
                tokens_chunk = set(corpus_tokenise[i])
                if tokens_question and set(tokens_question).issubset(tokens_chunk):
                    scores[i] += 1.0

        classement = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)
        rang_pertinent = next(
            (
                rang
                for rang, i in enumerate(classement, start=1)
                if exemple["passage_reference"].casefold() in chunks[i].casefold()
            ),
            None,
        )
        recall_hits.append(1 if rang_pertinent is not None and rang_pertinent <= 3 else 0)
        reciprocal_ranks.append(1 / rang_pertinent if rang_pertinent else 0)

    return {
        "Recall@3": sum(recall_hits) / len(recall_hits),
        "MRR": sum(reciprocal_ranks) / len(reciprocal_ranks),
        "Nombre de chunks": len(chunks),
    }


In [ ]:
resultats = []
for nom, chunks in strategies.items():
    mesures = evaluer(chunks)
    resultats.append({"Stratégie": nom, **mesures})

tableau = pd.DataFrame(resultats)
tableau.style.format({"Recall@3": "{:.3f}", "MRR": "{:.3f}"})


## 4. Comparer avec un re-ranking simple

Le bonus ci-dessous ajoute 1 point quand tous les mots de la question sont présents dans le chunk. Il s’agit d’un reranking lexical volontairement simple ; il ne comprend pas les synonymes.


In [ ]:
resultats_reranking = []
for nom, chunks in strategies.items():
    mesures = evaluer(chunks, reranking=True)
    resultats_reranking.append({"Stratégie": nom, **mesures})

tableau_reranking = pd.DataFrame(resultats_reranking)
tableau_reranking.style.format({"Recall@3": "{:.3f}", "MRR": "{:.3f}"})


In [ ]:
comparaison = tableau[["Stratégie", "Recall@3", "MRR"]].merge(
    tableau_reranking[["Stratégie", "Recall@3", "MRR"]],
    on="Stratégie",
    suffixes=(" sans bonus", " avec bonus"),
)
comparaison


## 5. Analyse des résultats

Les valeurs affichées dans les tableaux donnent le gagnant sur **ce texte et ces huit annotations**. Les métriques peuvent changer si l’on choisit d’autres questions ou une autre vérité terrain.

- Un meilleur **Recall@3** signifie que davantage de passages de réponse apparaissent dans les trois premiers résultats.
- Un meilleur **MRR** signifie que les premiers passages pertinents sont placés plus haut dans le classement.
- La stratégie par phrases/paragraphes préserve généralement mieux les unités de sens. Les blocs fixes peuvent couper une réponse au milieu d’une phrase.
- Le chevauchement de B peut récupérer un passage situé de part et d’autre d’une frontière : une partie de la réponse répétée dans le chunk suivant peut redevenir retrouvable. Il augmente toutefois le nombre de chunks et peut créer des résultats redondants.
- Le re-ranking peut aider quand les mots de la question apparaissent ensemble dans le bon chunk. Il peut aussi échouer quand la question reformule le passage avec des synonymes.

### Lecture automatique du gagnant


In [ ]:
gagnant = tableau.sort_values(["MRR", "Recall@3"], ascending=False).iloc[0]
print(
    f"Meilleur MRR sur ce jeu : {gagnant['Stratégie']} "
    f"(MRR={gagnant['MRR']:.3f}, Recall@3={gagnant['Recall@3']:.3f}, "
    f"{int(gagnant['Nombre de chunks'])} chunks)."
)

comparaison["Écart MRR (bonus - sans bonus)"] = (
    comparaison["MRR avec bonus"] - comparaison["MRR sans bonus"]
)
print("\nEffet du bonus sur le MRR :")
print(comparaison[["Stratégie", "Écart MRR (bonus - sans bonus)"]].to_string(index=False))
